# Consumption - Looping All Selected Countries and Run All Scenarios

## Purpose
This **master orchestration notebook** automates batch consumption forecasting by calling a worker notebook for each country-scenario combination.

---

## How It Works

**Orchestrator-Worker Pattern:**
```
THIS NOTEBOOK (Orchestrator)          WORKER NOTEBOOK
   ├─ Loops through countries    →    Receives ONE country + ONE scenario
   ├─ Loops through scenarios    →    Trains ARIMA/Prophet models
   └─ Adds metadata columns      →    Writes forecast to gold tables
```

**Worker Notebook:** [Consumption Forecast - Single Scenario by Country](#notebook-897483376288296)
* Receives parameters: `country`, `scenario`
* Timeout: 30 minutes per scenario
* Output: 6-year annual forecast + model metrics

---

## What This Notebook Does

1. **Cell 3**: Setup
   * Lists 20 countries to process
   * Queries available scenarios from `workspace.gold.exogenous_variables`
   * Filters for **Consumption + Baseline scenarios only** (Production excluded)
   * Estimates total runtime (~2 minutes per scenario)

2. **Cell 6**: Main Loop
   * Nested loop: `for country → for scenario → run worker notebook`
   * Progress tracking with real-time updates
   * Error handling: logs failures, continues processing

3. **Cell 7-8**: Post-Processing
   * Adds `generic_name` column (e.g., "COVID Shutdowns in Canada")
   * Adds `event_direction` column ('baseline' or 'Consumption')

---

## Data Sources & Output

**Input:**
* `workspace.gold.exogenous_variables` (scenario definitions)

**Output:**
* `workspace.gold.consumption_forecast_table_annual` (6-year forecasts with confidence intervals)
* `workspace.gold.model_metrics` (model performance per country-scenario)

---

## How to Run

### Full Batch (All 20 Countries)
1. Run **Cell 3** to see estimated time
2. Run **Cell 6** to start forecasting (monitor progress in real-time)
3. Run **Cells 7-8** to add metadata columns

### Test with Single Country
In Cell 3, uncomment:
```python
countries = ['Canada']  # Test with one country
```

**Expected Runtime:**
* Per scenario: ~2 minutes
* Full batch: Several hours (depends on scenarios per country)

---

## Key Features

* **Batch automation**: 20 countries × multiple scenarios
* **Scenario filtering**: Consumption + Baseline only
* **Error resilience**: Logs failures, continues processing
* **Progress tracking**: Real-time completion updates
* **Automatic metadata**: Adds descriptive columns after forecasting

---

## Notes

* Production scenarios are processed by a separate notebook: `Production - Run All Countries and Scenarios`
* If a scenario fails, check the error log in Cell 6 output and re-run for specific countries
* All forecasts use serverless compute (auto-selected)


### Define List of Countries & Estimate Total Processing Time

In [0]:
# Define list of all countries to process
countries = [
    'United States', 'Russia', 'Saudi Arabia', 'Canada', 'Iraq', 'China', 
    'Iran', 'United Arab Emirates', 'Brazil', 'Kuwait', 'Mexico', 'Kazakhstan', 
    'Nigeria', 'Norway', 'Angola', 'Venezuela', 'Qatar', 'Algeria', 'Japan', 'Singapore'
]

# #uncomment below, swap with above, to test run with only 1 country
# countries = ['Canada']

total_countries = len(countries)
approx_time_min = 2  # Assuming 2 minutes per scenario

print(f"{'='*80}")
print(f"STARTING BATCH PROCESSING FOR {total_countries} COUNTRIES")
print(f"-- Each scenario will take ~{approx_time_min} minutes to process --")
print(f"{'='*80}\n")

# Initialize total estimated time before the loop
total_estimated_time = 0

# Loop through each country
for country_idx, country in enumerate(countries, start=1):
    print(f"{'-'*70}")
    print(f"COUNTRY {country_idx}/{total_countries}: {country}")
        
    # Get all unique scenarios for the given country from workspace.gold.exogenous_variables (**only for CONSUMPTION & Baseline)
    scenario_list_df = spark.sql(f"""
    SELECT DISTINCT description, Event_Direction
    FROM workspace.gold.exogenous_variables
    WHERE country = '{country}' AND (Event_Direction = 'Consumption' OR Event_Direction = 'Baseline')
    """)
    
    if scenario_list_df.count() == 0:
        print(f"⚠ No scenarios found for {country}. Skipping...")
        continue
    
    # #uncomment to display the senario list of each country
    # scenario_list_df.display()
    
    # Calculate total scenarios and estimated time
    total_scenarios = scenario_list_df.count()
    estimated_minutes = total_scenarios * approx_time_min  
    
    print(f"{country} has {total_scenarios} scenarios to run. Estimated time: {estimated_minutes} minutes.")
    
    # Accumulate total estimated time
    total_estimated_time += estimated_minutes
    
    # #uncomment to display the completion message after displaying the scenarios
    #print(f"✓ Displayed scenarios for {country} ({country_idx}/{total_countries} countries done)")

print(f"\n\n{'='*80}")
# #uncomment to display the completion message after displaying all country scenarios
#print(f"✓ ALL {total_countries} COUNTRIES SCENARIOS DISPLAYED")
print(f"Total estimated time to run all scenarios is {total_estimated_time} minutes.")
print(f"{'='*80}")

STARTING BATCH PROCESSING FOR 20 COUNTRIES
-- Each scenario will take ~2 minutes to process --

----------------------------------------------------------------------
COUNTRY 1/20: United States
United States has 9 scenarios to run. Estimated time: 18 minutes.
----------------------------------------------------------------------
COUNTRY 2/20: Russia
Russia has 1 scenarios to run. Estimated time: 2 minutes.
----------------------------------------------------------------------
COUNTRY 3/20: Saudi Arabia
Saudi Arabia has 1 scenarios to run. Estimated time: 2 minutes.
----------------------------------------------------------------------
COUNTRY 4/20: Canada
Canada has 4 scenarios to run. Estimated time: 8 minutes.
----------------------------------------------------------------------
COUNTRY 5/20: Iraq
Iraq has 1 scenarios to run. Estimated time: 2 minutes.
----------------------------------------------------------------------
COUNTRY 6/20: China
China has 2 scenarios to run. Estimated 

In [0]:
# #assign the country variable
# country = 'Angola'

# #get all unique scenarios for the given country from workspace.gold.exogenous_variables
# scenario_list_df = spark.sql(f"""
# SELECT DISTINCT description, Event_Direction
# FROM workspace.gold.exogenous_variables
# WHERE country = '{country}'
# ORDER BY description
# """)

# scenario_list_df.display()

# # Calculate total scenarios and estimated time
# total_scenarios = scenario_list_df.count()
# estimated_minutes = total_scenarios * 10  # Assuming 10 minutes per scenario

# print(f"\n{country} has {total_scenarios} scenarios to run. Estimated time to run all scenarios is {estimated_minutes} minutes.")

### Run the Consumption Forecasts

In [0]:
# Loop through all countries and run consumption forecasts for each
import time

print(f"\n{'='*80}")
print(f"STARTING CONSUMPTION FORECASTS FOR ALL COUNTRIES")
print(f"{'='*80}\n")

# Initialize total time tracker before the loop
total_time_taken = 0

for country_idx, country in enumerate(countries, start=1):
    print(f"{'#'*70}")
    print(f"COUNTRY {country_idx}/{total_countries}: {country}")
    print(f"{'#'*70}\n")
    
    # Get all unique Consumption scenarios for the given country from workspace.gold.exogenous_variables
    scenario_list_df = spark.sql(f"""
    SELECT DISTINCT description
    FROM workspace.gold.exogenous_variables
    WHERE country = '{country}' AND (Event_Direction = 'Consumption' OR Event_Direction = 'Baseline')
    """)
    
    # Collect all scenarios into a list
    scenarios = [row.description for row in scenario_list_df.collect()]
    total_scenarios = len(scenarios)
    
    if total_scenarios == 0:
        print(f"⚠ No Consumption scenarios found for {country}. Skipping...")
        continue
    
    print(f"{'='*60}")
    print(f"Starting CONSUMPTION forecast loop for {country}")
    print(f"Total scenarios to process: {total_scenarios}")
    print(f"{'='*60}")
    
    # Record start time for this country
    start_time = time.time()

    # Loop through each scenario and run the forecast
    for idx, scenario in enumerate(scenarios, start=1):
        print(f"[{idx}/{total_scenarios}] Processing scenario: {scenario}")
        print("-" * 60)
        
        try:
            dbutils.notebook.run(
                '/Workspace/Users/mijimorgan@gmail.com/Modelling Workflow Scripts/2-Forecasting/Worker Notebooks/Consumption Forecast - Single Scenario by Country',
                1800,
                {'scenario': scenario, 'country': country}
            )
            print(f"✓ Completed {idx}/{total_scenarios} scenarios ({int(idx/total_scenarios*100)}% done)")
        except Exception as e:  # Error handling
            import traceback
            print(f"\n{'='*60}")
            print(f"❌ ERROR in scenario {idx}/{total_scenarios}: {scenario}")
            print(f"Error: {type(e).__name__}")
            print(f"\nParameters: scenario='{scenario}', country='{country}'")
            print(f"\nFull error details:")
            print(traceback.format_exc())
            print(f"{'='*60}\n")
    
    # Record end time and calculate time taken for this country
    end_time = time.time()
    time_taken = (end_time - start_time) / 60  # Convert to minutes
    
    # Accumulate total time
    total_time_taken += time_taken
    
    print(f"{'='*60}")
    print(f"✓ ALL CONSUMPTION SCENARIOS COMPLETED for {country}")
    print(f"Total scenarios processed: {total_scenarios}")
    print(f"Time taken for {country}: {time_taken:.2f} minutes")
    print(f"{'='*60}")
    print(f"\n✓ Completed country {country_idx}/{total_countries}\n")

print(f"\n{'='*80}")
print(f"✓ ALL CONSUMPTION FORECASTS COMPLETED FOR ALL {total_countries} COUNTRIES")
print(f"Total time taken: {total_time_taken:.2f} minutes ({total_time_taken/60:.2f} hours)")
print(f"{'='*80}")


STARTING CONSUMPTION FORECASTS FOR ALL COUNTRIES

######################################################################
COUNTRY 1/20: United States
######################################################################

Starting CONSUMPTION forecast loop for United States
Total scenarios to process: 9
[1/9] Processing scenario: Baseline scenario
------------------------------------------------------------
✓ Completed 1/9 scenarios (11% done)
[2/9] Processing scenario: Alberta Clipper Pipeline (Enbridge Line 67) completed
------------------------------------------------------------
✓ Completed 2/9 scenarios (22% done)
[3/9] Processing scenario: 1973 OPEC Oil Embargo
------------------------------------------------------------
✓ Completed 3/9 scenarios (33% done)
[4/9] Processing scenario: Original Enbridge Line 3 Pipeline completed
------------------------------------------------------------
✓ Completed 4/9 scenarios (44% done)
[5/9] Processing scenario: Express Pipeline completed to 

### Adding Other Required Descriptive Columns

In [0]:
# Add generic_name to forecast tables by copying from exogenous_variables
# generic_name is now pre-computed in exogenous_variables (by Update Binary Scenario Variables notebook)
# Just copy it directly - no need to compute CONCAT(Event_Type, ' in ', Country) here
# Join on: Country + Scenario (Description) + Event_Direction (Production/Consumption)
# Updates for ALL countries at once

print(f"\n{'='*80}")
print(f"ADDING GENERIC_NAME TO CONSUMPTION FORECAST TABLES")
print(f"{'='*80}\n")

# List of tables to update
tables = [
    'workspace.gold.consumption_forecast_table_annual'
]

for table_name in tables:
    print(f"Processing {table_name}...")
    
    # Check if generic_name column exists, if not add it
    table_schema = spark.table(table_name).schema
    column_names = [field.name for field in table_schema.fields]
    
    if 'generic_name' not in column_names:
        print(f"  Adding generic_name column...")
        spark.sql(f"""
            ALTER TABLE {table_name}
            ADD COLUMN generic_name STRING
        """)
        print(f"  ✓ Column added")
    else:
        print(f"  ✓ Column already exists")
    
    # Determine Event_Direction based on table name
    if 'production' in table_name:
        event_direction = 'Production'
    elif 'consumption' in table_name:
        event_direction = 'Consumption'
    else:
        event_direction = None
    
    # Copy generic_name directly from exogenous_variables
    # Match on Country + Scenario (Description in exogenous_variables) + Event_Direction
    print(f"  Copying generic_name values from exogenous_variables...")
    
    spark.sql(f"""
        MERGE INTO {table_name} AS forecast
        USING (
            SELECT DISTINCT
                Country,
                LOWER(REPLACE(Description, ' ', '_')) as Scenario_normalized,
                generic_name
            FROM workspace.gold.exogenous_variables
            WHERE Event_Direction = '{event_direction}'
              AND generic_name IS NOT NULL
        ) AS exo
        ON forecast.Country = exo.Country
           AND LOWER(REPLACE(forecast.Scenario, ' ', '_')) = exo.Scenario_normalized
        WHEN MATCHED THEN UPDATE SET
            forecast.generic_name = exo.generic_name
    """)
    
    # Verify the update
    verify_df = spark.sql(f"""
        SELECT 
            Country,
            Scenario,
            generic_name,
            COUNT(*) as row_count
        FROM {table_name}
        WHERE generic_name IS NOT NULL
        GROUP BY Country, Scenario, generic_name
        ORDER BY Country, Scenario
    """)
    
    updated_count = verify_df.count()
    
    if updated_count > 0:
        print(f"  ✓ Updated {updated_count} unique country+scenario combinations")
        print(f"  Sample of generic_name assignments:")
        sample_df = verify_df.limit(10)
        for row in sample_df.collect():
            print(f"    - {row.Country} | {row.Scenario} → {row.generic_name} ({row.row_count} rows)")
    else:
        print(f"  ℹ No scenarios matched for update")
    
    print()

print("="*80)
print(f"✓ GENERIC_NAME UPDATE COMPLETE FOR ALL COUNTRIES")
print("="*80)



ADDING GENERIC_NAME TO CONSUMPTION FORECAST TABLES

Processing workspace.gold.consumption_forecast_table_annual...
  ✓ Column already exists
  Copying generic_name values from exogenous_variables...
  ✓ Updated 10 unique country+scenario combinations
  Sample of generic_name assignments:
    - Canada | 1973_opec_oil_embargo → Export Cut in Canada (51 rows)
    - Canada | covid_shutdowns → Demand Disruption in Canada (51 rows)
    - Canada | enbridge_line_6_pipeline_completed → New Pipeline in Canada (51 rows)
    - China | covid_shutdowns → Demand Disruption in China (51 rows)
    - Japan | 1973_opec_oil_embargo → Export Cut in Japan (51 rows)
    - Japan | covid_shutdowns → Demand Disruption in Japan (51 rows)
    - Singapore | covid_shutdowns → Demand Disruption in Singapore (51 rows)
    - United States | 1973_opec_oil_embargo → Export Cut in United States (51 rows)
    - United States | covid_shutdowns → Demand Disruption in United States (51 rows)
    - United States | enbridge_l

In [0]:
# Add event_direction to consumption forecast tables for all countries
# For consumption notebook: 'baseline' if baseline_scenario, otherwise 'Consumption'

print(f"\n{'='*80}")
print(f"ADDING EVENT_DIRECTION TO CONSUMPTION FORECAST TABLES")
print(f"{'='*80}\n")

# List of consumption tables to update
tables = [
    'workspace.gold.consumption_forecast_table_annual',
]

for table_name in tables:
    print(f"Processing {table_name}...")
    
    # Check if event_direction column exists, if not add it
    table_schema = spark.table(table_name).schema
    column_names = [field.name for field in table_schema.fields]
    
    if 'event_direction' not in column_names:
        print(f"  Adding event_direction column...")
        spark.sql(f"""
            ALTER TABLE {table_name}
            ADD COLUMN event_direction STRING
        """)
        print(f"  ✓ Column added")
    else:
        print(f"  ✓ Column already exists")
    
    # Update event_direction based on scenario for ALL countries
    # baseline_scenario → 'baseline', all others → 'Consumption'
    print(f"  Updating event_direction values for all countries...")
    
    spark.sql(f"""
        UPDATE {table_name}
        SET event_direction = CASE 
            WHEN Scenario = 'baseline_scenario' THEN 'baseline'
            ELSE 'Consumption'
        END
    """)
    
    # Verify the update
    verify_df = spark.sql(f"""
        SELECT 
            Country,
            Scenario,
            event_direction,
            COUNT(*) as row_count
        FROM {table_name}
        GROUP BY Country, Scenario, event_direction
        ORDER BY Country, Scenario
    """)
    
    print(f"  ✓ Updated {verify_df.count()} unique country+scenario combinations")
    
    # Show sample
    print(f"  Sample of event_direction assignments:")
    sample_df = verify_df.limit(10)
    for row in sample_df.collect():
        print(f"    - {row.Country} | {row.Scenario} → {row.event_direction} ({row.row_count} rows)")
    
    print()

print("="*80)
print(f"✓ EVENT_DIRECTION UPDATE COMPLETE FOR ALL COUNTRIES")
print("="*80)


ADDING EVENT_DIRECTION TO CONSUMPTION FORECAST TABLES

Processing workspace.gold.consumption_forecast_table_annual...
  ✓ Column already exists
  Updating event_direction values for all countries...
  ✓ Updated 35 unique country+scenario combinations
  Sample of event_direction assignments:
    - Algeria | baseline_scenario → baseline (51 rows)
    - Angola | baseline_scenario → baseline (51 rows)
    - Brazil | baseline_scenario → baseline (51 rows)
    - Canada | 1973_opec_oil_embargo → Consumption (51 rows)
    - Canada | baseline_scenario → baseline (51 rows)
    - Canada | covid_shutdowns → Consumption (51 rows)
    - Canada | enbridge_line_6_pipeline_completed → Consumption (51 rows)
    - China | baseline_scenario → baseline (51 rows)
    - China | covid_shutdowns → Consumption (51 rows)
    - Iran | baseline_scenario → baseline (51 rows)

✓ EVENT_DIRECTION UPDATE COMPLETE FOR ALL COUNTRIES
